# 05 — Rolling DES + Baselines B0/B3/B4

Notebook này khóa kiến trúc simulation trước khi triển khai B6.

## Event model
- `IncidentCreated`
- `AmbulanceDispatched`
- `AmbulanceArrivesPatient`
- `TransportStarts`
- `HospitalArrival`
- `HandoverComplete`
- `ChargingStarts`
- `ChargingEnds`
- `AmbulanceAvailable`

## State riêng theo policy
Mỗi policy phải có bản sao riêng của:
- vehicle location;
- vehicle SOC;
- vehicle status;
- `available_at`;
- hospital runtime occupancy/capacity effect.

Không được dùng chung endogenous state giữa B0/B3/B4/B6.

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Tuple, List
import heapq
import pandas as pd
import numpy as np

@dataclass
class VehicleState:
    vehicle_id: str
    node: int | None
    lat: float
    lon: float
    soc_pct: float
    battery_capacity_kwh: float
    nominal_consumption_kwh_km: float
    reserve_soc_pct: float
    status: str
    available_at: pd.Timestamp

@dataclass
class HospitalState:
    hospital_id: str
    runtime_capacity: int
    estimated_wait_min: float

@dataclass(order=True)
class Event:
    timestamp: pd.Timestamp
    sequence: int
    event_type: str = field(compare=False)
    payload: dict = field(compare=False, default_factory=dict)

print("State/event contracts loaded.")

## Baseline definitions

### B0
Nearest/fastest available ambulance + nearest feasible destination under minimal hard safety rules.

### B3
Hospital-aware:
- capability;
- modeled capacity;
- travel time.

### B4
Energy-aware:
- B3 feasibility;
- SOC reserve;
- post-hospital charger reachability.

### B6
**Không triển khai trong notebook này.** B6 phải đi qua Notebook 06 và không được thay bằng arbitrary weighted-sum heuristic rồi vẫn gọi là B6.

In [ ]:
POLICY_CONTRACT = pd.DataFrame([
    {"policy_id":"B0","hospital_aware":False,"soc_aware":False,"uncertainty_aware":False},
    {"policy_id":"B3","hospital_aware":True, "soc_aware":False,"uncertainty_aware":False},
    {"policy_id":"B4","hospital_aware":True, "soc_aware":True, "uncertainty_aware":False},
    {"policy_id":"B6","hospital_aware":True, "soc_aware":True, "uncertainty_aware":True},
])
POLICY_CONTRACT

## Required DES invariants

1. `0 <= SOC <= 100`.
2. Xe `charging` không được dispatch.
3. Không double-dispatch một xe.
4. Event timestamps monotone.
5. `charging_completed => SOC_after >= SOC_before`.
6. Không dùng dữ liệu tương lai (`future leakage`).
7. B0/B3/B4/B6 cùng exogenous realization/seed nhưng state endogenous độc lập.

In [ ]:
INVARIANTS = [
    "SOC in [0,100]",
    "charging vehicle cannot be dispatched",
    "no vehicle double-dispatch",
    "event timestamps monotone",
    "charging completed => SOC_after >= SOC_before",
    "no future leakage",
    "policy-specific endogenous state"
]
pd.DataFrame({"invariant": INVARIANTS})

## Output contract trên mỗi policy × replication

- `dispatch_decisions.csv.gz`
- `des_event_log.csv.gz`
- `mission_metrics.csv.gz`
- `constraint_violations.csv`
- `replication_metrics.csv`

Notebook 07 chỉ được chạy final paired analysis khi các outputs này tồn tại cho cả B0/B3/B4/B6.